# Google Trends in Colab without 429 errors: compare any number of keywords on one scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sassrami/google-trends-compare/blob/main/google_trends_colab.ipynb)

pytrends often fails in Colab with `TooManyRequestsError` / HTTP 429 on the very first request: Colab runs on Google Cloud IPs, and Google Trends blocks most of them. pytrends is also archived, and Google compares at most 5 keywords per request, each request rescaled to its own 0-100.

This notebook calls a hosted Google Trends scraper on Apify instead. It handles sessions and residential proxies, and puts **any number of keywords on one shared 0-100 scale**.

**Cost:** $0.005 per keyword (one result per keyword's time series). The 8 keywords below cost about $0.04, which Apify's free monthly credit covers. Keywords with no data are not charged.

*Disclosure: I built and maintain the scraper ([granola2/google-trends-scraper](https://apify.com/granola2/google-trends-scraper)).*

In [ ]:
!pip install -q apify-client

## 1. Your Apify token

Create a free account at [apify.com](https://apify.com) and copy your API token (Settings > API & Integrations). In Colab, click the key icon in the left sidebar, add a secret named `APIFY_TOKEN`, and allow this notebook to use it. Otherwise the next cell asks for it.

In [ ]:
import getpass
import os

try:
    from google.colab import userdata

    token = userdata.get("APIFY_TOKEN")
except Exception:  # not in Colab, or the secret is not set
    token = os.environ.get("APIFY_TOKEN") or getpass.getpass("Apify API token: ")

## 2. Pick keywords

Any number. More than 5 are stitched onto one scale automatically.

In [ ]:
KEYWORDS = ["chatgpt", "gemini", "claude", "perplexity", "copilot", "deepseek", "grok", "cursor"]
GEO = "US"  # country code, "" for worldwide
TIME = "today 12-m"  # also "today 5-y", "now 7-d", "2024-01-01 2024-12-31", ...
# Or paste a link from trends.google.com instead: set TRENDS_URL and it overrides the three above.
TRENDS_URL = ""

In [ ]:
from apify_client import ApifyClient

client = ApifyClient(token)
run_input = {"trendsUrl": TRENDS_URL} if TRENDS_URL else {"keywords": KEYWORDS, "geo": GEO, "timeRange": TIME}
run = client.actor("granola2/google-trends-scraper").call(run_input={**run_input, "reports": ["interestOverTime"]})
items = client.dataset(run.default_dataset_id).list_items().items
print(f"Run {run.status}: {len(items)} keywords returned")

## 3. Ranking and chart

`rank` and `shareOfTotalPct` compare the keywords directly, because they all share one scale.

In [ ]:
import pandas as pd

series = [i for i in items if i.get("report") == "interestOverTime"]
ranking = pd.DataFrame(
    [{k: s.get(k) for k in ("rank", "keyword", "averageValue", "peakValue", "shareOfTotalPct")} for s in series]
).set_index("rank").sort_index()
df = pd.DataFrame({s["keyword"]: {p["date"]: p["value"] for p in s["points"]} for s in series}).sort_index()
df.index = pd.to_datetime(df.index).rename("date")
ranking

In [ ]:
ax = df.plot(figsize=(12, 6), logy=True, title="Google Trends interest, one shared scale (log)")
ax.set_ylabel("interest (biggest keyword peaks at 100)");

## 4. Download the data

In [ ]:
df.to_csv("trends.csv")
try:
    from google.colab import files

    files.download("trends.csv")
except ImportError:
    print("Saved trends.csv")